# Channel reconstruction — checkpoint validation

Evaluate any checkpoint produced by `train_channel.py` with recorded train and validation splits. Change `CHECKPOINT` in the configuration cell to a different S3 checkpoint URI or a local `.pt` file. The notebook uses the checkpoint's architecture, field mapping, wall, target depth, spatial stride, and validation slice. The default is the `channel_full3_ptov/dainty-sunset-0` best checkpoint. This is validation, not a held-out test of checkpoint selection.

Run from the project root or `networks/full` with the project environment and S3 read access. `RUN_MODE = 'off'` is safe. Use `'normalize'` to prepare or check the normalization without reading validation fields, or `'evaluate'` to generate predictions and compare with the recorded validation split. No training or S3 writes occur.

Normalization matches `train_channel.py`: one mean and sample standard deviation per input/target channel, computed only from the exact training slice. Files `xmeans.npy`, `xstds.npy`, `ymeans.npy`, and `ystds.npy` are saved under `networks/full/.channel_norm_cache/<key>/`, with a provenance manifest. Matching runs reuse them regardless of checkpoint name. A changed source, slice, or grid selection has a different key. If source content is replaced in place without a metadata change, move the relevant cache directory aside before recomputing. The older top-level `ymeans.npy` and `ystds.npy` are **per-y profiles for `full_2.ipynb`**; they are not compatible with this trainer and are not overwritten or used here.

A cache miss will not read training fields unless `ALLOW_COMPUTE_NORMALIZATION = True`. For the default checkpoint, one pass over A1 `[200:3196:1]` (2,996 snapshots) reads **107.31 GB**; validation `re200-sr1etot` `[200:1001:1]` (801 snapshots) reads **28.69 GB** plus another **28.69 GB** for exact full-sample PDFs if enabled. Other checkpoints can have different costs. Only reductions are retained in memory between batches.

In [1]:
from pathlib import Path
from io import BytesIO
import sys
import hashlib
import json
import tempfile
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

for root in (Path.cwd(), Path.cwd() / 'networks' / 'full'):
    if (root / 'train_channel.py').is_file():
        sys.path.insert(0, str(root))
        break
else:
    raise FileNotFoundError('Start the kernel from the project root or networks/full')

from train_channel import (S3_BUCKET, S3_ENDPOINT, TASK_CHANNELS, _s3fs, _open_data_zarr,
                           _open_channel_axis, make_model)
SOURCE_DIR = root

COMPONENTS = ('u', 'v', 'w')
STRESS_PAIRS = ((0, 1, 'u′v′'), (0, 2, 'u′w′'), (1, 2, 'v′w′'))

In [2]:
WALL_PRESSURE_TO_VELOCITY = dict(in_ch=3, out_start=0, out_ch=3, wall_index=0)


def read_batch(a, s, ny, stride, selection=WALL_PRESSURE_TO_VELOCITY):
    i, o, c, wall = (selection[k] for k in ('in_ch','out_start','out_ch','wall_index'))
    depth = ny if wall is None else max(ny, wall+1)
    # Read each snapshot chunk once when the input and targets share a chunk.
    f = np.asarray(a[s, :max(i+1,o+c), ::stride, :depth, ::stride], dtype=np.float32)
    iy = slice(0,ny) if wall is None else slice(wall,wall+1)
    return np.ascontiguousarray(f[:, i:i+1, :, iy, :]), np.ascontiguousarray(f[:, o:o+c, :, :ny, :])


def selections(r, batch_size):
    if batch_size < 1 or r.step < 1 or not len(r):
        raise ValueError('Need a nonempty forward selection and positive batch size')
    for i in range(0, len(r), batch_size):
        b = r[i:i + batch_size]
        yield slice(b.start, b.stop, b.step)


def training_norm(a, r, ny, stride, batch_size, progress=False, selection=WALL_PRESSURE_TO_VELOCITY):
    # Parallel-variance merge; equivalent to train_channel._channel_stats (ddof=1).
    counts = [0, 0]
    means = [0., 0.]
    m2s = [0., 0.]
    for s in tqdm(selections(r, batch_size), total=(len(r)+batch_size-1)//batch_size,
                  desc='A1 normalization', disable=not progress):
        for j, v in enumerate(read_batch(a, s, ny, stride, selection=selection)):
            if not np.isfinite(v).all():
                raise ValueError(f'Nonfinite training field in {s}')
            dims = (0, 2, 3, 4)
            n = v.size // v.shape[1]
            mean = np.mean(v, axis=dims, dtype=np.float64)
            m2 = np.var(v, axis=dims, dtype=np.float64) * n
            delta = mean - means[j]
            total = counts[j] + n
            m2s[j] = m2s[j] + m2 + delta**2 * (counts[j] * n / total)
            means[j] = means[j] + delta * (n / total)
            counts[j] = total
    if min(counts) < 2:
        raise ValueError("Training normalization needs at least two values per channel")
    def pack(v):
        return np.asarray(v, dtype=np.float32)[None, :, None, None, None]
    return tuple(v for j in range(2) for v in
                 (pack(means[j]), pack(np.sqrt(m2s[j]/(counts[j]-1))).clip(min=1e-6)))


def prediction_batches(a, r, ny, stride, batch_size, model, norm, device, progress=False, selection=WALL_PRESSURE_TO_VELOCITY):
    xm, xs, ym, ys = (torch.as_tensor(q, dtype=torch.float32, device=device) for q in norm)
    for s in tqdm(selections(r, batch_size), total=(len(r)+batch_size-1)//batch_size,
                  desc='Validation inference', disable=not progress):
        x, target = read_batch(a, s, ny, stride, selection=selection)
        with torch.inference_mode():
            prediction = model((torch.from_numpy(x).to(device) - xm) / xs)
            physical = (prediction * ys + ym).cpu().numpy()
        yield physical, target


NORM_FILES = ('xmeans.npy','xstds.npy','ymeans.npy','ystds.npy')


def cached_norm(root, metadata, allow_compute, compute):
    """Write an entire cache entry at once and validate it on both write and reuse."""
    manifest = json.dumps(metadata,sort_keys=True,indent=2) + '\n'
    entry = Path(root) / hashlib.sha256(manifest.encode()).hexdigest()[:24]

    def checked(values):
        widths = (1,1,metadata['n_out'],metadata['n_out'])
        for i,(v,width) in enumerate(zip(values,widths)):
            if (v.dtype != np.float32 or v.shape != (1,width,1,1,1) or not np.isfinite(v).all()
                or (i%2 and np.any(v < np.float32(1e-6)))):
                raise ValueError(f'Invalid cache array: {NORM_FILES[i]}')
        return values

    def load():
        try:
            if (entry/'manifest.json').read_text() != manifest:
                raise ValueError('manifest mismatch')
            return checked(tuple(np.load(entry/name,allow_pickle=False) for name in NORM_FILES))
        except (OSError,ValueError,EOFError) as exc:
            raise ValueError(f'Invalid normalization cache {entry}; move it aside before rebuilding') from exc

    if entry.exists():
        return load(),entry
    if not allow_compute:
        raise FileNotFoundError(f'No normalization cache at {entry}; approve the exact training read before enabling ALLOW_COMPUTE_NORMALIZATION')
    values = checked(compute())
    entry.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory(dir=entry.parent,prefix='.pending-') as tmp:
        staged = Path(tmp)/'entry'; staged.mkdir()
        for name,value in zip(NORM_FILES,values,strict=True):
            np.save(staged/name,value,allow_pickle=False)
        (staged/'manifest.json').write_text(manifest)
        try:
            staged.rename(entry)
        except OSError:
            if not entry.is_dir(): raise
    return load(),entry

## Metrics and wall layers

- **Pointwise comparison:** global and component MSE; plane-wise MSE, mean and population standard deviation (fluctuation RMS) versus distance from the measured wall; three Reynolds shear stress profiles; fluctuation PDFs for all evaluated planes and one buffer-layer plane, with buffer-plane skewness and excess kurtosis. Profiles and stresses pool snapshots and x/z points; stress is a mean, not the unnormalized sum in the Couette notebook. No interpolation or wall folding. Volume-input runs use distance from the lower wall; pressure-input runs use distance from the measured wall. Scalar `Q` runs use the same scalar diagnostics without velocity-only Reynolds shear stress.
- **Spectra:** full x–z FFT using the sampled grid and periodic lengths $L_x=N_x\Delta x$ and $L_z=N_z\Delta z$ (**12.57** and **4.19** for the default run). Use physical radial wavenumber $|k|$, with annuli centered on multiples of $\Delta k=\min(2\pi/L_x,2\pi/L_z)$, half-bin edges, and the zero mode reported separately. Each plane and snapshot has equal weight within its layer, matching Couette's sample pooling on a nonuniform y grid (not a wall-normal volume integral).
- **Two spectral errors:** squared *power* mismatch, as in the Couette eval, averaged over modes, planes and snapshots per bin (the legacy sums and relative ratio are also available in `results`); and **paired Fourier MSE** from $|\hat u_{pred}-\hat u_{DNS}|^2$, which retains phase errors. The latter sums to the physical MSE over all bins including DC. Raw-field spectra include the mean; DC is excluded from the radial plots but printed. The paired relative error is separately normalized by DNS power, with near-zero denominators masked.
- **Layer boundaries:** nominal $y^+=(y-y_{wall})Re_\tau/\delta$, using the supplied `RE_TAU` (200 for the example run) and the grid half-height $\delta$. Diagnostics split into viscous sublayer $[0,5)$, buffer layer $[5,30)$, nominal log/overlap band $[30,100)$, and outer/core $[100,\infty)$. `RE_TAU` can be changed if you have a measured value. These boundaries are diagnostic bins, not a claim of a developed log law at this Reynolds number. See the project `validation.md` for the near-wall and nominal log ranges.

In [3]:
def fft_grid(nx, nz, lx, lz):
    kx = 2*np.pi*np.fft.fftfreq(nx, d=lx/nx)
    kz = 2*np.pi*np.fft.fftfreq(nz, d=lz/nz)
    dk = min(2*np.pi/lx, 2*np.pi/lz)
    r = np.hypot(kx[:, None], kz[None, :])
    bins = np.maximum(1, np.floor(r/dk + .5).astype(int))
    bins[0, 0] = 0
    count = np.bincount(bins.ravel())
    return dict(bins=bins.ravel(), count=count, k=np.arange(count.size)*dk,
                dk=dk, nx=nx, nz=nz)


def relative(numer, denom):
    scale = denom.max(axis=-1, keepdims=True)
    return np.divide(numer, denom, out=np.full_like(numer, np.nan),
                     where=denom > np.maximum(scale*1e-14, 0.0))


def evaluate(factory, grid, masks, pdf_plane, do_pdf=True):
    nx, nz, count = grid['nx'], grid['nz'], grid['count']
    ny = len(next(iter(masks.values())))
    layers = {'all': np.ones(ny, dtype=bool), **masks}
    snapshots = 0
    for p, t in factory():
        if not snapshots:
            nc = t.shape[1]
            mse = np.zeros((nc,ny)); sum1 = np.zeros((2,nc,ny))
            sum2 = np.zeros((2,nc,nc,ny))
            extrema = np.full_like(sum1,np.inf), np.full_like(sum1,-np.inf)
            bands = {key:np.zeros((4,nc,count.size)) for key,sel in layers.items() if sel.any()}
        if p.shape != t.shape or p.ndim != 5 or p.shape[1:] != (nc,nx,ny,nz):
            raise ValueError(f'Unexpected prediction/truth grid: {p.shape}, {t.shape}')
        if not (np.isfinite(p).all() and np.isfinite(t).all()):
            raise ValueError(f'Nonfinite velocity near validation sample {snapshots}')
        p, t = (np.asarray(v, dtype=np.float64) for v in (p,t))
        snapshots += len(t)
        mse += ((p-t)**2).sum(axis=(0,2,4))
        for j, v in enumerate((p,t)):
            sum1[j] += v.sum(axis=(0,2,4))
            sum2[j] += np.einsum('tcxyz,tdxyz->cdy',v,v)
            np.minimum(extrema[0][j], v.min(axis=(0,2,4)), out=extrema[0][j])
            np.maximum(extrema[1][j], v.max(axis=(0,2,4)), out=extrema[1][j])
        for c in range(nc):
            f, g = (np.fft.fft2(v[:,c], axes=(1,3))/(nx*nz) for v in (p,t))
            pp, tp = abs(f)**2, abs(g)**2
            for i, power in enumerate((pp, tp, (pp-tp)**2, abs(f-g)**2)):
                per_y = power.sum(axis=0)  # (X,Y,Z)
                for name, out in bands.items():
                    mode = per_y[:,layers[name],:].sum(axis=1)
                    out[i,c] += np.bincount(grid['bins'], weights=mode.ravel(), minlength=count.size)
    if not snapshots:
        raise ValueError('No validation predictions')
    npts = snapshots*nx*nz
    mse /= npts
    means = sum1/npts
    stress = sum2/npts - means[:,:,None,:]*means[:,None,:,:]
    std = np.sqrt(np.maximum(0,np.diagonal(stress, axis1=1, axis2=2).transpose(0,2,1)))
    spectra = {}
    for name,out in bands.items():
        ny_layer = int(layers[name].sum())
        n = snapshots*ny_layer
        pred, dns, sq, paired = out/n
        np.testing.assert_allclose(paired.sum(axis=-1), mse[:,layers[name]].mean(axis=1),
                                   rtol=1e-7, atol=1e-14, err_msg=f'Parseval: {name}')
        spectra[name] = dict(pred=pred, dns=dns, paired=paired, paired_rel=relative(paired,dns),
            power_mse=np.divide(sq,count,out=np.full_like(sq,np.nan),where=count>0),
            power_sq_sum=out[2], planes=ny_layer,
            couette_raw_sum=out[2]*(nx*nz)**2, couette_relative=relative(out[2]*(nx*nz),out[1]))
    result = dict(snapshots=snapshots, mse_y=mse, mse_component=mse.mean(axis=1),
                  mse_global=mse.mean(), mean=means, stress=stress, std=std, spectra=spectra, pdf=None)
    if not do_pdf:
        return result
    # Exactly match the full-data, common symmetric PDF bins; do not store all predictions.
    lo, hi = (v-means for v in extrema)
    limits = np.maximum(abs(lo),abs(hi)).max(axis=0)
    pdf = []
    for ys in ((slice(None),pdf_plane) if pdf_plane is not None else (slice(None),)):
        entries = []
        for c in range(nc):
            lim = float(np.max(limits[c,ys]))*1.05 or 1.0
            entries.append(dict(edges=np.linspace(-lim,lim,120),counts=np.zeros((2,119),dtype=np.int64),
                                shape_sum=np.zeros((2,2))))
        pdf.append(entries)
    n2 = 0
    for p,t in factory():
        n2 += len(t)
        for j,v in enumerate((p,t)):
            for c in range(nc):
                for i,ys in enumerate((slice(None),pdf_plane) if pdf_plane is not None else (slice(None),)):
                    d = pdf[i][c]
                    fluc = v[:,c,:,ys,:] - (means[j,c][None,None,:,None] if i==0 else means[j,c,ys])
                    d['counts'][j] += np.histogram(fluc, bins=d['edges'])[0]
                    if i==1:
                        d['shape_sum'][j] += ((fluc**3).sum(), (fluc**4).sum())
    if n2 != snapshots:
        raise ValueError('Validation pass counts differ')
    for i,entries in enumerate(pdf):
        for c,d in enumerate(entries):
            if not np.all(d['counts'].sum(axis=-1)==npts*(ny if i==0 else 1)):
                raise ValueError('PDF histogram lost samples')
            if i==1:
                den = std[:,c,pdf_plane,None] ** np.array([3,4])
                d['shape'] = np.divide(d['shape_sum']/npts,den,out=np.full_like(den,np.nan),where=den>0) - [0,3]
    result['pdf'] = pdf
    return result

In [4]:
def plot_results(m, y, yp, grid, pdf_plane, components=COMPONENTS):
    nc = len(components)
    scale = float(yp.max()/y.max())
    print(f"Validation: {m['snapshots']} snapshots; global physical MSE = {m['mse_global']:.6e}")
    print('Component MSE:', dict(zip(components, m['mse_component'])))
    metrics = ('mean','std','mse_y','stress') if nc == 3 else ('mean','std','mse_y')
    for metric in metrics:
        fig, axs = plt.subplots(1,nc,figsize=(max(5,4*nc),3.2),layout='constrained',sharex=True,squeeze=False)
        fig.suptitle({'mean':'Mean field','std':'Population std / fluctuation RMS',
                      'mse_y':'Physical MSE by measured-wall distance', 'stress':'Reynolds shear stress'}[metric])
        for c, ax in enumerate(axs[0]):
            title = components[c]
            if metric == 'mse_y':
                ax.plot(y,m[metric][c]); ax.set_ylabel('MSE (field units²)')
            else:
                if metric == 'stress':
                    i,j,title = STRESS_PAIRS[c]; pred,dns = m[metric][:,i,j]
                else:
                    pred,dns = m[metric][:,c]
                ax.plot(y,dns,label='DNS'); ax.plot(y,pred,label='Prediction')
                ax.plot(y,pred-dns,'k--',label='Prediction − DNS'); ax.legend(fontsize=8)
            ax.set(title=title,xlabel='Distance from measured wall'); ax.grid(alpha=.25)
            sec = ax.secondary_xaxis('top',functions=(lambda a:a*scale,lambda a:a/scale))
            sec.set_xlabel('Nominal y⁺')
        plt.show()
    valid = (grid['count']>0) & (grid['k']>0)
    x = grid['k'][valid]
    specs = m['spectra']
    labels = ('DNS and predicted power', 'Power-spectrum MSE', 'Paired Fourier MSE', 'Paired MSE / DNS power')
    fig, axs = plt.subplots(len(specs),4,figsize=(15,2.7*len(specs)),layout='constrained',squeeze=False)
    for row,(layer,s) in enumerate(specs.items()):
        print(f"{layer}: {s['planes']} planes; DC paired MSE: {s['paired'][:,0]}")
        for col,ax in enumerate(axs[row]):
            for c,(name,color) in enumerate(zip(components,('C0','C1','C2'))):
                if col == 0:
                    ax.plot(x,s['dns'][c,valid],color=color,label=f'{name}: DNS')
                    ax.plot(x,s['pred'][c,valid],'--',color=color,label=f'{name}: pred')
                else:
                    key = ('power_mse','paired','paired_rel')[col-1]
                    ax.plot(x,s[key][c,valid],color=color,label=name)
            if any(np.any(np.asarray(line.get_ydata()) > 0) for line in ax.lines):
                ax.set_yscale('log')
            ax.set(title=labels[col] if row==0 else '',xlabel='|k| (rad / length)')
            if col==0: ax.set_ylabel(layer)
            if row==0: ax.legend(fontsize=7,ncol=2 if col==0 else 1)
            ax.grid(alpha=.25)
    plt.show()
    if m['pdf'] is not None:
        fig, axs = plt.subplots(nc,len(m['pdf']),figsize=(11,max(3,2.7*nc)),layout='constrained',squeeze=False)
        fig.suptitle('Field fluctuation PDFs')
        for col,entries in enumerate(m['pdf']):
            for c,d in enumerate(entries):
                ax = axs[c,col]
                for j,name in enumerate(('Prediction','DNS')):
                    h,e = d['counts'][j],d['edges']
                    ax.stairs(h/(h.sum()*np.diff(e)),e,label=name)
                loc = 'all evaluated planes' if col==0 else f'buffer plane y⁺={yp[pdf_plane]:.1f}'
                ax.set(title=f'{components[c]} — {loc}',xlabel='Fluctuation',ylabel='Density'); ax.legend()
                if col==1: print(f"{components[c]} skew, excess kurtosis (prediction / DNS): {d['shape'].tolist()}")
        plt.show()

In [5]:
CHECKPOINT = 's3://models/channel-1/channel_full3_ptov/dainty-sunset-0/ckpt_best.pt'
RUN_MODE = 'off'  # 'normalize': cache only; 'evaluate': run validation; 'off': no network I/O.
ALLOW_COMPUTE_NORMALIZATION = False  # Allow a training read on a cache miss only.
CACHE_ROOT = SOURCE_DIR / '.channel_norm_cache'
VALIDATION_LIMIT = None
EXACT_PDFS = True
BATCH_SIZE = 2
RE_TAU = 200.0  # Set from the chosen run's validation DNS.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def checkpoint_split(record):
    prefix = f's3://{S3_BUCKET}/'
    uri = record['uri']
    if not uri.startswith(prefix) or not uri.endswith('/data.zarr'):
        raise ValueError(f'Not a project channel data URI: {uri}')
    r = range(*record['slice'])
    if not len(r) or r.step < 1 or r.start < 0 or len(r) != record['snapshots']:
        raise ValueError(f'Invalid checkpoint slice: {record}')
    return uri[len(prefix):-len('/data.zarr')], r


def spatial_selection(cfg, a):
    i,o,c = TASK_CHANNELS[cfg['task']]
    ny = (a.shape[3]+1)//2 if cfg['half_y'] else a.shape[3]
    wall = cfg['wall']
    if wall not in (None,'lower','upper') or len(a.shape)!=5 or max(i+1,o+c)>a.shape[1]:
        raise ValueError('Invalid wall or array layout')
    stride = cfg['xz_stride']
    if ny != cfg['arch']['dy'] or c != cfg['arch']['out_ch'] or stride < 1:
        raise ValueError('Checkpoint architecture does not match the source geometry')
    wall_index = {'lower':0,'upper':a.shape[3]-1,None:None}[wall]
    return ny,stride,dict(in_ch=i,out_start=o,out_ch=c,wall_index=wall_index)


def axes_for(source, array):
    axes = [_open_channel_axis(source,a) for a in 'xyz']
    for axis,size in zip(axes,array.shape[2:]):
        if axis.shape != (size,) or not np.isfinite(axis).all() or not np.all(np.diff(axis)>0):
            raise ValueError(f'Invalid grid at {source}')
    return axes


results = None
if RUN_MODE not in ('off','normalize','evaluate'):
    raise ValueError('Unknown RUN_MODE')
if RUN_MODE != 'off':
    cp = BytesIO(_s3fs().cat_file(CHECKPOINT.removeprefix('s3://'))) if CHECKPOINT.startswith('s3://') else Path(CHECKPOINT).expanduser()
    checkpoint = torch.load(cp,map_location='cpu',weights_only=True)
    cfg = checkpoint['config']
    ds = cfg['data_selection']
    train_root,tr = checkpoint_split(ds['train'])
    ta = _open_data_zarr(train_root)
    if tr[-1] >= ta.shape[0]: raise ValueError('Training selection exceeds array')
    ny,stride,selection = spatial_selection(cfg,ta)
    train_axes = axes_for(train_root,ta)
    meta = dict(algorithm='train_channel_global_float32_ddof1_1e-6_v1',
                uri=ds['train']['uri'],slice=ds['train']['slice'],endpoint=S3_ENDPOINT,
                dtype=str(ta.dtype),shape=list(ta.shape[1:]),target_depth=ny,
                stride=stride, n_out=selection['out_ch'], geometry=selection,
                grid_sha256=[hashlib.sha256(axis.astype('<f8').tobytes()).hexdigest() for axis in train_axes])
    norm,cache_dir = cached_norm(CACHE_ROOT,meta,ALLOW_COMPUTE_NORMALIZATION,
        lambda: training_norm(ta,tr,ny,stride,BATCH_SIZE,progress=True,selection=selection))
    print(f'Training normalization: {cache_dir.resolve()}')
    if RUN_MODE == 'evaluate':
        val_root,vr = checkpoint_split(ds['validation'])
        va = _open_data_zarr(val_root)
        if ta.shape[1:] != va.shape[1:] or vr[-1]>=va.shape[0]: raise ValueError('Validation shape or time mismatch')
        if VALIDATION_LIMIT is not None:
            if not 1 <= VALIDATION_LIMIT <= len(vr): raise ValueError('Invalid validation limit')
            vr = vr[:VALIDATION_LIMIT]
        axes = axes_for(val_root,va)
        for axis,ref in zip(axes,train_axes): np.testing.assert_allclose(axis,ref,rtol=1e-7,atol=1e-10)
        model = make_model(cfg['arch'],DEVICE).eval()
        model.load_state_dict(checkpoint['model_state_dict'],strict=True)
        x, full_y, z = axes
        for a in (x,z): np.testing.assert_allclose(np.diff(a),np.diff(a).mean(),rtol=1e-6)
        if any(len(a)%stride for a in (x,z)):
            raise ValueError('An FFT requires the stride to evenly divide the periodic grid')
        if not np.isfinite(RE_TAU) or RE_TAU <= 0: raise ValueError('RE_TAU must be positive and finite')
        lx,lz = len(x)*np.diff(x).mean(),len(z)*np.diff(z).mean()
        wall = selection['wall_index'] if selection['wall_index'] is not None else 0
        y = abs(full_y[:ny] - full_y[wall])
        yp = y * RE_TAU / ((full_y[-1]-full_y[0])/2)
        edges = (0,5,30,100,np.inf)
        names = ('Viscous sublayer','Buffer layer','Nominal log/overlap','Outer/core')
        masks = {name:(yp>=l)&(yp<h) for name,l,h in zip(names,edges,edges[1:])}
        print('Wall layer plane counts:', {k:int(v.sum()) for k,v in masks.items()})
        buffer = np.flatnonzero(masks['Buffer layer'])
        pdf_plane = buffer[np.argmin(abs(yp[buffer]-15))] if len(buffer) else None
        if EXACT_PDFS and pdf_plane is None: print('No buffer plane in target grid; plotting the all-plane PDF only.')
        grid = fft_grid(len(x[::stride]),len(z[::stride]),lx,lz)
        print(f'Box lengths: {lx:.6g}, {lz:.6g}; Δk={grid["dk"]:.6g}; validation {vr}')
        factory = lambda: prediction_batches(va,vr,ny,stride,BATCH_SIZE,model,norm,DEVICE,progress=True,selection=selection)
        results = evaluate(factory,grid,masks,pdf_plane,EXACT_PDFS)
        if results['snapshots'] != len(vr): raise ValueError('Validation snapshot count mismatch')
        labels = COMPONENTS if selection['out_ch'] == 3 else ('Q',)
        results.update(y=y,y_plus=yp,k=grid['k'],k_mode_count=grid['count'],lx=lx,lz=lz,
                       validation_indices=np.fromiter(vr,dtype=int),component_labels=labels,
                       checkpoint=CHECKPOINT,best_epoch_index=cfg.get('best_epoch'),layer_masks=masks,
                       normalization_cache=str(cache_dir.resolve()))
else:
    print('Live data disabled. Select a run mode after approving the read scope.')

Wall layer plane counts: {'Viscous sublayer': 5, 'Buffer layer': 20, 'Nominal log/overlap': 26, 'Outer/core': 14}
Box lengths: 12.57, 4.19; Δk=0.499856; validation range(200, 1001); best epoch index 19


A1 normalization:   0%|          | 0/1498 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
if results is not None:
    plot_results(results,y,yp,grid,pdf_plane,results['component_labels'])

## Compare saved metrics across runs

Add a run name to `COMPARE_RUNS` (within `COMPARE_PROJECT`) and run this section. Leave the single-run `RUN_MODE` above set to `'off'` when using this section. The comparison pulls verified metrics from `s3://prediction/<project>/<run>/metrics/`; matching runs always reuse their Zarr results. When `COMPUTE_MISSING_METRICS` is enabled, only missing runs go through inference, using the checkpoint’s exact training normalization cache. Missing runs share each DNS read. Checkpoints and DNS remain read-only. The exact PDFs use temporary local paired fields, not a second DNS or inference pass; the persisted data are the metrics, coordinates and snapshot IDs, not prediction volumes.

Run comparisons start disabled to avoid an accidental large read when you use *Run All*. Set `SHOW_COMPARISON = True` to load metrics; then set `COMPUTE_MISSING_METRICS = True` if new runs should be evaluated automatically on a cache miss. Full comparison uses the common validation split without subsampling. Mismatched targets, input walls, grids, or validation selections are rejected. `EXACT_PDFS` and `RE_TAU` are part of the cache key; keep them fixed when adding comparable runs. With both default runs missing, allow about 29 GB of S3 reads and 31 GB of temporary space (use `COMPARE_SCRATCH` to choose a suitable disk). The required bucket already exists on the project S3 service.

In [ ]:
from channel_eval_compare import compare_channel_runs, plot_comparison

COMPARE_PROJECT = 'channel_full3_ptov'
COMPARE_RUNS = ['dainty-sunset-0', 'brave-forest-1']
SHOW_COMPARISON = False
COMPUTE_MISSING_METRICS = False
COMPARE_SCRATCH = None  # Or a path to an existing directory on a disk with enough space.

comparison = None
if SHOW_COMPARISON:
    comparison = compare_channel_runs(
        COMPARE_RUNS, project=COMPARE_PROJECT,
        compute_missing=COMPUTE_MISSING_METRICS, exact_pdfs=EXACT_PDFS,
        re_tau=RE_TAU, batch_size=1, scratch=COMPARE_SCRATCH, device=DEVICE,
    )
    for run, m in comparison.items():
        print(f"{run}: {m['snapshots']} snapshots; MSE={m['mse_global']:.6g}; "
              f"per component={dict(zip(m['component_labels'],m['mse_component'].tolist()))}")
    comparison_figures = plot_comparison(comparison)
    plt.show()

The Zarr results also include the original single-run metrics (mean, RMS, Reynolds stresses, PDFs, and spectra). To show those for any entry already in `comparison`, run:

```python
m = comparison['brave-forest-1']
plot_results(m, m['y'], m['y_plus'], {'k': m['k'], 'count': m['k_mode_count']},
             m['pdf_plane'], m['component_labels'])
```